## Letter confounder tests

In [1]:
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import rankdata

plt.rcParams.update({'font.size': 12})
LETTERS = ['B','C','D','F','G','H','K','L','N','P','R','S','T','V','Z']
L_IDX   = {l: i for i, l in enumerate(LETTERS)}
PAD     = 'X'

SIG_LETTERS = ['F','B','G','C','V','D','N','R']     # FDR p<.05, see 2c_letter_decoding
sig = np.array([l in SIG_LETTERS for l in LETTERS])
print('decoded     :', [l for l in LETTERS if sig[L_IDX[l]]])
print('not decoded :', [l for l in LETTERS if not sig[L_IDX[l]]])

decoded     : ['B', 'C', 'D', 'F', 'G', 'N', 'R', 'V']
not decoded : ['H', 'K', 'L', 'P', 'S', 'T', 'Z']


## Load data

In [2]:
info    = np.load('../Data/trial_corrInfo.npy', allow_pickle=True)
letters = np.load('../Data/letters_sep.npy',    allow_pickle=True)
probes  = np.load('../Data/probes_sep.npy',     allow_pickle=True)
subjs   = np.load('../Data/subjs_sep.npy',      allow_pickle=True)

subj    = np.concatenate(subjs, axis=0)
rawL    = [list(s) for s in np.concatenate(letters, axis=0)]
rawP    = [list(s) for s in np.concatenate(probes,  axis=0)]
infoALL = np.concatenate(info)
infoALL = infoALL[infoALL[:, 1] == 1]
assert len(rawL) == len(infoALL) == len(subj)

# ordered memory string: strip the X placeholders, order is preserved
strings = [[c for c in s if c != PAD] for s in rawL]
probe1  = [s[0] if len(s) else None for s in rawP]
is_IN   = np.array([int(r[3]) == 51 for r in infoALL])

## Per-letter properties

In [3]:
n_trials = np.zeros(15)     # trials in which the letter appeared
size_sum = np.zeros(15)     # sum of set sizes of the trials it appeared in

for s in strings:
    L = len(s)
    for c in s:
        if c not in L_IDX:          # outside the 15-consonant pool
            continue
        j = L_IDX[c]
        n_trials[j] += 1
        size_sum[j] += L

props = pd.DataFrame({
    'decoded':       sig,
    'n_trials':      n_trials,
    'pct_of_trials': 100 * n_trials / len(strings),
    'mean_set_size': size_sum / n_trials,
}, index=LETTERS)
print(props.round(4).to_string())

   decoded  n_trials  pct_of_trials  mean_set_size
B     True     976.0        37.3517         6.0143
C     True     940.0        35.9740         6.0362
D     True     944.0        36.1271         6.0847
F     True    1003.0        38.3850         6.0399
G     True     951.0        36.3949         6.0252
H    False     933.0        35.7061         6.0043
K    False     957.0        36.6246         6.0752
L    False    1012.0        38.7294         6.0000
N     True     930.0        35.5913         6.0602
P    False     975.0        37.3134         6.0615
R     True     955.0        36.5480         6.0126
S    False     939.0        35.9357         6.0980
T    False     894.0        34.2135         6.0694
V     True     968.0        37.0455         5.9773
Z    False     935.0        35.7826         5.9872


## Phonological features

The same 17-dimensional IPA feature matrix used in `Table1_phonological.ipynb`.

In [4]:
FEAT_NAMES = ['voiced','bilabial','labiodental','alveolar','velar','glottal','affricate',
              'fricative','nasal','stop','lateral','trill','vowel_onset',
              'rhyme_ee','rhyme_aa','rhyme_eps','rhyme_au']
FEAT = np.array([
 [1,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0],   # B
 [0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0],   # C
 [1,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0],   # D
 [0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,1,0],   # F
 [1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0],   # G
 [0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0],   # H
 [0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0],   # K
 [1,0,0,1,0,0,0,0,0,0,1,0,1,0,0,1,0],   # L
 [1,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1,0],   # N
 [0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0],   # P
 [1,0,0,1,0,0,0,0,0,0,0,1,1,0,0,1,0],   # R
 [0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,1,0],   # S
 [0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0],   # T
 [0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1],   # V
 [0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0]],  # Z
 dtype=float)
_N  = FEAT / np.linalg.norm(FEAT, axis=1, keepdims=True)
COS = _N @ _N.T

## Permutation tests

In [5]:
SPLITS = [np.isin(np.arange(15), s) for s in itertools.combinations(range(15), 8)]

def _mean_diff(v, m): return v[m].mean() - v[~m].mean()
def _ranksum(v, m):   return rankdata(v)[m].sum()

def exact_test(values, stat='mean'):
    v = np.asarray(values, dtype=float)
    f = _mean_diff if stat == 'mean' else _ranksum
    obs  = f(v, sig)
    null = np.array([f(v, m) for m in SPLITS])
    centre = null.mean()
    p = np.mean(np.abs(null - centre) >= abs(obs - centre) - 1e-12)
    return obs, p

def run_family(table, name):
    rows = []
    for col, v in table.items():
        v = np.asarray(v, dtype=float)
        d_mean, p_mean = exact_test(v, 'mean')
        _,      p_rank = exact_test(v, 'rank')
        rows.append({'property': col,
                     'decoded': v[sig].mean(), 'not_decoded': v[~sig].mean(),
                     'difference': d_mean, 'p_mean': p_mean, 'p_rank': p_rank})
    out = pd.DataFrame(rows)
    print(f'--- {name} ({len(out)} tests) ---')
    print(out.round(4).to_string(index=False))
    print()
    return out

## Phonological features — one test per feature

In [6]:
ALPHA = .05

def min_attainable_p(values, stat='mean'):
    """Smallest two-sided p this property could return under any labelling of the 15 letters.

    The null distribution depends only on `values`, so the most extreme achievable
    result is the fraction of splits at the maximum distance from the null centre.
    """
    v    = np.asarray(values, dtype=float)
    f    = _mean_diff if stat == 'mean' else _ranksum
    null = np.array([f(v, m) for m in SPLITS])
    dev  = np.abs(null - null.mean())
    return np.mean(dev >= dev.max() - 1e-12)

def holm(p):
    p = np.asarray(p, dtype=float); k = len(p); o = np.argsort(p)
    adj = np.maximum.accumulate((k - np.arange(k)) * p[o])
    out = np.empty(k); out[o] = np.minimum(adj, 1.0)
    return out

feats   = {nm: FEAT[:, k] for k, nm in enumerate(FEAT_NAMES)}
pmin    = {nm: min_attainable_p(v) for nm, v in feats.items()}
usable  = {nm: v for nm, v in feats.items() if pmin[nm] < ALPHA}
dropped = [nm for nm in FEAT_NAMES if nm not in usable]

print(f'{len(dropped)} of {len(feats)} features cannot reach p < {ALPHA} under any labelling:')
print(pd.DataFrame({'n_letters':       [int(feats[nm].sum()) for nm in dropped],
                    'best_possible_p': [round(pmin[nm], 4)   for nm in dropped]},
                   index=dropped).to_string(), '\n')

phon = run_family(usable, f'phonological features (informative only, {len(usable)}/{len(feats)})')

10 of 17 features cannot reach p < 0.05 under any labelling:
             n_letters  best_possible_p
bilabial             2           0.2000
labiodental          2           0.2000
velar                2           0.2000
glottal              1           0.4667
affricate            2           0.2000
nasal                1           0.4667
lateral              1           0.4667
trill                1           0.4667
rhyme_aa             2           0.2000
rhyme_au             1           0.4667 

--- phonological features (informative only, 7/17) (7 tests) ---
   property  decoded  not_decoded  difference  p_mean  p_rank
     voiced    0.625       0.1429      0.4821  0.1189  0.1189
   alveolar    0.500       0.5714     -0.0714  1.0000  1.0000
  fricative    0.250       0.2857     -0.0357  1.0000  1.0000
       stop    0.375       0.4286     -0.0536  1.0000  1.0000
vowel_onset    0.375       0.2857      0.0893  1.0000  1.0000
   rhyme_ee    0.500       0.2857      0.2143  0.6084  0.608

## Multivariate: do the decoded letters cluster in phonological space?


In [7]:
def cohesion(mask):
    iu = lambda m: COS[np.ix_(m, m)][np.triu_indices(m.sum(), k=1)]
    within  = np.concatenate([iu(mask), iu(~mask)])
    between = COS[np.ix_(mask, ~mask)].ravel()
    return within.mean() - between.mean()

obs_c  = cohesion(sig)
null_c = np.array([cohesion(m) for m in SPLITS])
p_c    = np.mean(null_c >= obs_c - 1e-12)          # one-sided: more cohesive than chance
print(f'within-group minus between-group cosine similarity = {obs_c:+.4f}')
print(f'  null mean {null_c.mean():+.4f}, SD {null_c.std():.4f}')
print(f'  exact permutation p (one-sided) = {p_c:.4f}')

within-group minus between-group cosine similarity = -0.0241
  null mean +0.0000, SD 0.0500
  exact permutation p (one-sided) = 0.6242


## Exposure

In [8]:
expo = run_family({'n_trials':      props.n_trials,
                   'mean_set_size': props.mean_set_size},
                  'exposure')

--- exposure (2 tests) ---
     property  decoded  not_decoded  difference  p_mean  p_rank
     n_trials 958.3750     949.2857      9.0893  0.5826  0.5358
mean_set_size   6.0313       6.0422     -0.0109  0.5846  0.6943



## Summary

In [9]:
allres = pd.concat([phon.assign(family='phonological'),
                    expo.assign(family='exposure')], ignore_index=True)

print('tests with p < .05:')
hit = allres[allres.p_mean < .05]
print(hit[['family','property','decoded','not_decoded',
           'difference','p_mean','p_rank']].round(4).to_string(index=False)
      if len(hit) else '   none')

tests with p < .05:
   none
